# Notebook 03: Data Cleaning

## I. Giới thiệu
Mục tiêu: Biến dữ liệu sau khi tích hợp từ PostgreSQL thành dữ liệu sạch, nhất quán và sẵn sàng cho EDA.
Pipeline: Read SQL → Missing Values → Duplicate → Outlier → Logic Validation → Save Clean Data → Notebook 04


## II. Đọc dữ liệu


In [ ]:
import pandas as pd
import numpy as np
import os

# Đọc dữ liệu từ DB (hoặc đọc tạm từ CSV nếu chưa setup xong DB)
# df = pd.read_sql("SELECT * FROM vw_tokyo_temp", engine)

df = pd.read_csv('../data/raw/GlobalLandTemperaturesByMajorCity.csv')
df['dt'] = pd.to_datetime(df['dt'])
city_df = df[df['City'] == 'Tokyo'].copy()
city_df = city_df.set_index('dt').sort_index()

# Đảm bảo trục thời gian liên tục từ năm 1850 đến 2013
full_idx = pd.date_range(start='1850-01-01', end=city_df.index.max(), freq='MS')
city_df = city_df.reindex(full_idx)
print(city_df.shape)


## III. Làm sạch dữ liệu
### 1. Xử lý Missing Values
Kiểm tra dữ liệu bị thiếu:


In [ ]:
missing_before = city_df.isnull().sum()
print("Missing before:\n", missing_before)

# Xử lý: Lấp đầy khoảng trống dữ liệu bằng Nội suy Time/Spline
city_df['AverageTemperature'] = city_df['AverageTemperature'].interpolate(method='time')
# Xóa các dòng NaN còn lại
city_df = city_df.dropna()

missing_after = city_df.isnull().sum()
print("Missing after:\n", missing_after)


Missing before:
 AverageTemperature               5
AverageTemperatureUncertainty    5
City                             0
Country                          0
Latitude                         0
Longitude                        0
dtype: int64
Missing after:
 AverageTemperature               0
AverageTemperatureUncertainty    0
City                             0
Country                          0
Latitude                         0
Longitude                        0
dtype: int64


### 2. Xử lý Duplicate
Kiểm tra bản ghi trùng lặp


In [ ]:
duplicates = city_df.duplicated().sum()
print("Số bản ghi trùng lặp:", duplicates)
if duplicates > 0:
    city_df = city_df.drop_duplicates()


Số bản ghi trùng lặp: 0


### 3. Xử lý Outlier và Logic
Kiểm tra các giá trị bất thường (ví dụ: nhiệt độ quá cao hoặc quá thấp).


In [ ]:
city_df['AverageTemperature'].describe()


count    1960.000000
mean       12.567906
std         8.235346
min        -1.580000
25%         4.525000
50%        13.162500
75%        20.177000
max        27.295000
Name: AverageTemperature, dtype: float64

## V. Đánh giá sau Cleaning
So sánh trước và sau làm sạch (số dòng, số cột, missing values). Dữ liệu hoàn toàn sạch sẽ.


## VI. Lưu dữ liệu sạch
Lưu lại dữ liệu đã làm sạch vào cơ sở dữ liệu để sử dụng cho bước tiếp theo.


In [ ]:
# Lưu vào PostgreSQL
# city_df.to_sql('tokyo_cleaned', engine, if_exists='replace')

# Lưu tạm ra CSV (dự phòng)
os.makedirs('../data/processed', exist_ok=True)
city_df.to_csv('../data/processed/tokyo_cleaned.csv')


## VII. Kết luận
Dữ liệu đã được làm sạch hoàn toàn (xử lý nội suy chuỗi thời gian, xóa NaN, loại bỏ duplicates) và sẵn sàng cho phần EDA.
